# 🌅 ArtiFact — Real vs AI-Generated Image Detection (EfficientNetV2B0)

**Model:** EfficientNetV2B0 + Custom Classification Head (Transfer Learning)

**Dataset:** [ArtiFact](https://github.com/awsaf49/artifact) — 2.5M images from 25 generators + 8 real sources

**Features:**
- ✅ Google Colab T4/A100 GPU optimized
- ✅ Google Drive auto-checkpoint (disconnect-safe)
- ✅ 2-Phase training (head → fine-tune top 40 layers)
- ✅ AdamW optimizer + Label Smoothing + L2 Regularization
- ✅ Mixed Precision (float16) for 2x speed
- ✅ **NO `.cache()`** — RAM crash prevention
- ✅ Auto-resume from any epoch after disconnect
- ✅ **Colab Disk Crash Safe** — Selective extraction parses and extracts only the balanced training subset (saves ~30GB of disk space!)
- ✅ **Multi-Account Resume Support** — Auto-saves and loads Kaggle credentials directly from Drive. Easily switch accounts to resume training.

**Expected Accuracy:** 93–97%+

---
### ⚡ Quick Start
1. `Runtime` → `Change runtime type` → Select **T4 GPU**
2. Run **Cell 1** (Mount Drive + Install)
3. Run **Cell 2** (Download Dataset — loads `kaggle.json` automatically from Drive if present)
4. Run **Cell 3** (Prepare Dataset — selective extraction into real/fake)
5. Run **All remaining cells** or `Runtime → Run all`

💡 **Multi-Account Note:** If you are switching Google accounts to bypass GPU limits: Share the `ArtiFact_Model` folder from your main account to your secondary account. On the secondary account, right-click the shared folder and select **"Add shortcut to Drive"**, placing it in your main **My Drive** directory. This ensures the paths match perfectly.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1: Mount Google Drive + Check GPU + Install Packages
# ─────────────────────────────────────────────────────────────────────────────
import os, sys

from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT_DIR = '/content/drive/MyDrive/ArtiFact_Model'
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
print(f'✅ Google Drive mounted. Project folder: {DRIVE_PROJECT_DIR}')

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'\nGPU Available: {len(gpus) > 0}')
if gpus:
    gpu_info = tf.config.experimental.get_device_details(gpus[0])
    gpu_name = gpu_info.get('device_name', 'Unknown')
    print(f'   GPU: {gpu_name}')
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy('mixed_float16')
    print('   Mixed Precision (float16): ENABLED ✅')
else:
    print('⚠️  No GPU! Go to Runtime > Change runtime type > T4 GPU')

print(f'\nTensorFlow version: {tf.__version__}')

!pip install -q scikit-learn seaborn
print('✅ All packages installed.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2: Download ArtiFact Dataset via Kaggle API (Auto-loads credentials)
# ─────────────────────────────────────────────────────────────────────────────
import os

# Official ArtiFact dataset (ICIP 2023 by awsaf49)
KAGGLE_SLUG = 'awsaf49/artifact-dataset'
DATASET_DIR = '/content/dataset'
PREPARED_DIR = '/content/prepared_dataset'

# Check if already prepared
already_prepared = False
if os.path.exists(PREPARED_DIR):
    r_dir = os.path.join(PREPARED_DIR, 'real')
    f_dir = os.path.join(PREPARED_DIR, 'fake')
    if os.path.exists(r_dir) and os.path.exists(f_dir):
        if len(os.listdir(r_dir)) > 0 and len(os.listdir(f_dir)) > 0:
            already_prepared = True

if already_prepared:
    print('✅ Dataset already prepared. Skipping download.')
else:
    print('Setting up Kaggle credentials...')
    kaggle_drive_path = os.path.join(DRIVE_PROJECT_DIR, 'kaggle.json')
    !mkdir -p ~/.kaggle

    if os.path.exists(kaggle_drive_path):
        print("✅ Found kaggle.json in Google Drive. Copying to environment...")
        !cp {kaggle_drive_path} ~/.kaggle/
    else:
        print('Kaggle credentials not found in Google Drive.')
        print('   Please upload your kaggle.json file when prompted.')
        print('   (Get it from: kaggle.com > Profile > Settings > API > Create New Token)\n')
        from google.colab import files
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            !cp kaggle.json ~/.kaggle/
            !cp kaggle.json {kaggle_drive_path}
            print(f"✅ Saved kaggle.json to Google Drive for future sessions: {kaggle_drive_path}")

    !chmod 600 ~/.kaggle/kaggle.json

    os.makedirs(DATASET_DIR, exist_ok=True)
    print(f'\nDownloading: {KAGGLE_SLUG} ...')
    !kaggle datasets download -d {KAGGLE_SLUG} -p {DATASET_DIR}
    print('✅ ZIP Download complete!')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3: Prepare Dataset — Selective ZIP Extraction (Colab Disk Crash Safe!)
# ─────────────────────────────────────────────────────────────────────────────
#
# The complete ArtiFact ZIP is 31.58 GB. Extracting the whole archive requires
# ~64GB of space, which will crash Colab free tier due to "disk full" errors.
# This cell scans the ZIP contents and extracts ONLY the balanced subset we need
# directly to its final location, then deletes the ZIP, using only ~1.5 GB of disk!
#
import os, random, gc, glob, zipfile
from pathlib import Path

PREPARED_DIR = '/content/prepared_dataset'
MAX_PER_CLASS = 50000  # Balanced subset size per class (Adjust as desired)

# Known real-image dataset names
KNOWN_REAL = {
    'ffhq', 'celebahq', 'celeba', 'coco', 'lsun', 'imagenet',
    'places', 'afhq', 'metfaces', 'real', 'factual', 'flickr',
    'laion', 'wiki', 'landscapes', 'sun', 'caltech', 'pascal',
    'cityscapes', 'openimages', 'inaturalist'
}
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

real_dir = os.path.join(PREPARED_DIR, 'real')
fake_dir = os.path.join(PREPARED_DIR, 'fake')

if os.path.exists(real_dir) and os.path.exists(fake_dir) and len(os.listdir(real_dir)) > 0:
    print(f'✅ Already prepared: {len(os.listdir(real_dir)):,} real, {len(os.listdir(fake_dir)):,} fake')
else:
    print('Scanning ZIP contents to classify real vs fake sources...')
    zip_files = glob.glob(os.path.join(DATASET_DIR, '*.zip'))
    if not zip_files:
        raise FileNotFoundError(f"No ZIP archive found under {DATASET_DIR}. Run Cell 2 first.")
    zip_path = zip_files[0]

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        namelist = zip_ref.namelist()
        img_members = [f for f in namelist if Path(f).suffix.lower() in IMG_EXTS]

        # Group by source directory
        source_map = {}
        for member in img_members:
            parts = Path(member).parts
            if len(parts) < 2:
                continue
            src_name = None
            for part in parts[:-1]:
                if part.lower() != 'artifact':
                    src_name = part
                    break
            if src_name:
                if src_name not in source_map:
                    source_map[src_name] = []
                source_map[src_name].append(member)

        # Classify each source folder as real or fake
        all_real_members = []
        all_fake_members = []

        print(f'\n{"Source":<30} {"Images":>8}  Label')
        print('-' * 50)
        for name, members in sorted(source_map.items()):
            clean = name.lower().replace('-', '').replace('_', '')
            is_real = any(r in clean for r in KNOWN_REAL)
            label = 'REAL' if is_real else 'FAKE'

            if is_real:
                all_real_members.extend(members)
            else:
                all_fake_members.extend(members)
            print(f'  {name:<28} {len(members):>8,}  {label}')

        print(f'\n  {"TOTAL REAL IN ZIP":<28} {len(all_real_members):>8,}')
        print(f'  {"TOTAL FAKE IN ZIP":<28} {len(all_fake_members):>8,}')

        if len(all_real_members) == 0 or len(all_fake_members) == 0:
            raise RuntimeError('Cannot train with 0 images in a class. Check classification mappings.')

        # Balance & limit dataset
        random.seed(42)
        if len(all_real_members) > MAX_PER_CLASS:
            all_real_members = random.sample(all_real_members, MAX_PER_CLASS)
        if len(all_fake_members) > MAX_PER_CLASS:
            all_fake_members = random.sample(all_fake_members, MAX_PER_CLASS)

        min_count = min(len(all_real_members), len(all_fake_members))
        all_real_members = all_real_members[:min_count]
        all_fake_members = all_fake_members[:min_count]

        print(f'\nExtracting balanced subset ({min_count:,} real, {min_count:,} fake) directly to output folders...')
        os.makedirs(real_dir, exist_ok=True)
        os.makedirs(fake_dir, exist_ok=True)

        for i, member in enumerate(all_real_members):
            ext = Path(member).suffix
            target = os.path.join(real_dir, f'real_{i:06d}{ext}')
            with open(target, 'wb') as f:
                f.write(zip_ref.read(member))
            if (i + 1) % 10000 == 0:
                print(f'   real: {i+1:,}/{min_count:,}')

        for i, member in enumerate(all_fake_members):
            ext = Path(member).suffix
            target = os.path.join(fake_dir, f'fake_{i:06d}{ext}')
            with open(target, 'wb') as f:
                f.write(zip_ref.read(member))
            if (i + 1) % 10000 == 0:
                print(f'   fake: {i+1:,}/{min_count:,}')

    # Remove ZIP file to save disk space
    try:
        os.remove(zip_path)
        print('\n✅ Extracted dataset successfully and deleted the raw ZIP archive!')
    except Exception as e:
        print(f'\n⚠️ Failed to delete ZIP: {e}')
    
    gc.collect()

# Verify counts
for label in ['real', 'fake']:
    p = os.path.join(PREPARED_DIR, label)
    count = len(os.listdir(p)) if os.path.exists(p) else 0
    print(f'   {label}: {count:,} images')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4: Configuration — Paths & Hyperparameters
# ─────────────────────────────────────────────────────────────────────────────
import os

# Cell 3 creates PREPARED_DIR with real/ and fake/ subdirs
BASE_DIR = '/content/prepared_dataset'

# Google Drive paths (survive disconnects)
DRIVE_DIR        = '/content/drive/MyDrive/ArtiFact_Model'
CHECKPOINT_DIR   = os.path.join(DRIVE_DIR, 'checkpoints')
MODEL_SAVE_PATH  = os.path.join(DRIVE_DIR, 'artifact_efficientnetv2b0.keras')
STATE_FILE       = os.path.join(CHECKPOINT_DIR, 'training_state.json')
HISTORY_FILE     = os.path.join(DRIVE_DIR, 'training_history.json')
PLOT_PATH        = os.path.join(DRIVE_DIR, 'training_history.png')
CLASS_NAMES_FILE = os.path.join(DRIVE_DIR, 'class_names.json')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Hyperparameters
IMAGE_SIZE    = (224, 224)
BATCH_SIZE    = 32
SEED          = 42

EPOCHS_HEAD   = 10
EPOCHS_FINE   = 25

LR_HEAD       = 1e-3
LR_FINE       = 1e-5

DROPOUT_RATE  = 0.4
L2_WEIGHT     = 1e-4
WEIGHT_DECAY  = 1e-4

UNFREEZE_LAYERS = 40

print('✅ Configuration loaded:')
print(f'   Image Size     : {IMAGE_SIZE}')
print(f'   Batch Size     : {BATCH_SIZE}')
print(f'   Phase 1 epochs : {EPOCHS_HEAD} (head only, LR={LR_HEAD})')
print(f'   Phase 2 epochs : {EPOCHS_FINE} (fine-tune top {UNFREEZE_LAYERS}, LR={LR_FINE})')
print(f'   Total epochs   : {EPOCHS_HEAD + EPOCHS_FINE}')
print(f'   Dropout        : {DROPOUT_RATE}')
print(f'   L2 reg         : {L2_WEIGHT}')
print(f'   AdamW decay    : {WEIGHT_DECAY}')
print(f'\n   Checkpoint Dir : {CHECKPOINT_DIR}')
print(f'   Final Model    : {MODEL_SAVE_PATH}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5: Helper Functions (Resume + State Tracking + Memory)
# ─────────────────────────────────────────────────────────────────────────────
import json, glob, re, gc
import tensorflow as tf


def find_latest_checkpoint(phase):
    """Find latest checkpoint for a given phase."""
    pattern = os.path.join(CHECKPOINT_DIR, f'artifact_{phase}_epoch_*.keras')
    files = glob.glob(pattern)
    if not files:
        return None, 0
    def get_epoch(f):
        m = re.search(r'epoch_(\d+)\.keras', os.path.basename(f))
        return int(m.group(1)) if m else 0
    files.sort(key=get_epoch)
    latest = files[-1]
    return latest, get_epoch(latest)


def load_state():
    """Load training state from Google Drive."""
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE, 'r') as f:
            state = json.load(f)
        print(f'Resumed state from Drive: {state}')
        return state
    print('No previous state found. Starting from scratch.')
    return {'phase': 'p1', 'last_epoch': 0}


def save_state(phase, epoch, val_acc=None, val_loss=None):
    """Save training state to Google Drive."""
    state = {'phase': phase, 'last_epoch': epoch}
    if val_acc is not None:
        state['best_val_acc'] = round(float(val_acc), 6)
    if val_loss is not None:
        state['best_val_loss'] = round(float(val_loss), 6)
    with open(STATE_FILE, 'w') as f:
        json.dump(state, f, indent=2)


class DriveStateLogger(tf.keras.callbacks.Callback):
    """Saves state to Google Drive after every epoch.
    Ensures training can resume after Colab disconnect."""
    def __init__(self, phase):
        super().__init__()
        self.phase = phase

    def on_epoch_end(self, epoch, logs=None):
        # In Keras, epoch is already absolute (accounts for initial_epoch)
        abs_epoch = epoch + 1  # convert 0-indexed to 1-indexed
        val_acc  = logs.get('val_accuracy') if logs else None
        val_loss = logs.get('val_loss') if logs else None
        save_state(self.phase, abs_epoch, val_acc, val_loss)
        acc_str = f'{val_acc:.4f}' if val_acc else 'N/A'
        print(f'  Drive saved -> phase={self.phase}, epoch={abs_epoch}, val_acc={acc_str}')


class RAMMonitor(tf.keras.callbacks.Callback):
    """Runs gc.collect() after every epoch to prevent RAM buildup."""
    def on_epoch_end(self, epoch, logs=None):
        gc.collect()


print('✅ Helper functions loaded.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6: Data Loading + Augmentation Pipeline
# ─────────────────────────────────────────────────────────────────────────────
#
# KEY: NO .cache() is used anywhere!
# Only .prefetch() -> prevents RAM crashes.
#
import tensorflow as tf
from tensorflow.keras import layers
import json, gc

AUTOTUNE = tf.data.AUTOTUNE

# Data Augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.04),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.2),
    layers.RandomTranslation(0.1, 0.1),
], name='augmentation')

# Load datasets: auto-split 80/10/10 from prepared real/fake dirs
print('Loading datasets...')

# 80% train
train_ds = tf.keras.utils.image_dataset_from_directory(
    BASE_DIR,
    validation_split=0.2,
    subset='training',
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
    shuffle=True
)
# 20% val+test
val_test_ds = tf.keras.utils.image_dataset_from_directory(
    BASE_DIR,
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
    shuffle=False
)

class_names = train_ds.class_names
print(f'✅ Classes: {class_names}')

# Save class_names to Drive for use on resume
with open(CLASS_NAMES_FILE, 'w') as f:
    json.dump(class_names, f)

# IMPORTANT: Save cardinality BEFORE transforms
# (after .map/.take/.skip, cardinality may return -2)
train_batch_count    = tf.data.experimental.cardinality(train_ds).numpy()
val_test_batch_count = tf.data.experimental.cardinality(val_test_ds).numpy()

# Split val+test 50/50 -> 10% val + 10% test of total
val_batches = val_test_batch_count // 2
val_ds  = val_test_ds.take(val_batches)
test_ds = val_test_ds.skip(val_batches)
del val_test_ds

print(f'   Train batches : {train_batch_count} (~{train_batch_count * BATCH_SIZE:,} images)')
print(f'   Val batches   : {val_batches} (~{val_batches * BATCH_SIZE:,} images)')
print(f'   Test batches  : {val_test_batch_count - val_batches} (~{(val_test_batch_count - val_batches) * BATCH_SIZE:,} images)')

# Pipeline: Augment (train only) + Prefetch (NO CACHE!)
train_ds = (
    train_ds
    .map(lambda x, y: (data_augmentation(x, training=True), y),
         num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)
val_ds  = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

gc.collect()
print('✅ Data pipeline ready. (NO .cache() -> RAM safe)')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7: Build Model — EfficientNetV2B0 + Custom Head
# ─────────────────────────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras import layers


def build_model():
    base_model = tf.keras.applications.EfficientNetV2B0(
        include_top=False,
        weights='imagenet',
        input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3),
        pooling=None
    )
    base_model.trainable = False

    inputs = tf.keras.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))

    x = tf.keras.applications.efficientnet_v2.preprocess_input(inputs)
    x = base_model(x, training=False)

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(256, kernel_regularizer=tf.keras.regularizers.l2(L2_WEIGHT))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(DROPOUT_RATE)(x)

    x = layers.Dense(128, kernel_regularizer=tf.keras.regularizers.l2(L2_WEIGHT))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(1, activation='sigmoid', dtype='float32')(x)

    model = tf.keras.Model(inputs, outputs, name='EfficientNetV2B0_ArtiFact')
    return model, base_model


model, base_model = build_model()

trainable = sum(1 for l in model.layers if l.trainable)
total     = len(model.layers)
print(f'✅ Model built: EfficientNetV2B0')
print(f'   Total layers    : {total}')
print(f'   Trainable layers: {trainable} (head only)')
print(f'   Total params    : {model.count_params():,}')
print(f'   Base layers     : {len(base_model.layers)}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8: PHASE 1 — Train Classifier Head Only
# (Base frozen, AdamW lr=1e-3, fast convergence)
# ─────────────────────────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras import callbacks
import gc

current_state = load_state()

p1_ckpt, p1_ckpt_epoch = find_latest_checkpoint('p1')
p1_initial_epoch = (
    current_state['last_epoch'] if current_state['phase'] == 'p1'
    else EPOCHS_HEAD  # p2, p3, done -> p1 is complete
)

history_p1 = None

if p1_initial_epoch >= EPOCHS_HEAD:
    print(f'✅ Phase 1 already complete ({p1_initial_epoch}/{EPOCHS_HEAD} epochs).')
    p1_final = os.path.join(CHECKPOINT_DIR, 'artifact_p1_final.keras')
    if p1_ckpt:
        model.load_weights(p1_ckpt)
        print(f'   Loaded weights: {os.path.basename(p1_ckpt)}')
    elif os.path.exists(p1_final):
        model.load_weights(p1_final)
        print(f'   Loaded final weights: artifact_p1_final.keras')
else:
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(
            learning_rate=LR_HEAD,
            weight_decay=WEIGHT_DECAY
        ),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )

    # Recovery fallback if state file is lost but checkpoints exist
    if p1_ckpt and p1_initial_epoch > 0:
        model.load_weights(p1_ckpt)
        print(f'Resuming Phase 1 from epoch {p1_initial_epoch}')
    elif p1_ckpt and p1_ckpt_epoch > 0:
        model.load_weights(p1_ckpt)
        p1_initial_epoch = p1_ckpt_epoch
        print(f'Resuming Phase 1 from checkpoint epoch {p1_initial_epoch}')
    else:
        print('Starting Phase 1 from scratch.')

    cb_p1 = [
        callbacks.ModelCheckpoint(
            filepath=os.path.join(CHECKPOINT_DIR, 'artifact_p1_epoch_{epoch:02d}.keras'),
            save_best_only=True,
            monitor='val_accuracy',
            mode='max',
            verbose=1
        ),
        callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=4,
            restore_best_weights=True,
            verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=1
        ),
        DriveStateLogger(phase='p1'),
        RAMMonitor(),
    ]

    print(f'\n=== PHASE 1: Train Head Only ===')
    print(f'    AdamW LR={LR_HEAD} | Epochs: {p1_initial_epoch+1} -> {EPOCHS_HEAD}')
    history_p1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_HEAD,
        initial_epoch=p1_initial_epoch,
        callbacks=cb_p1
    )

    # Always save final weights
    model.save_weights(os.path.join(CHECKPOINT_DIR, 'artifact_p1_final.keras'))
    save_state('p2', 0)
    gc.collect()
    print('✅ Phase 1 complete.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9: PHASE 2 — Fine-tune Top 40 Layers
# (Partial unfreeze, CosineDecay lr=1e-5)
# ─────────────────────────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras import callbacks
import gc

# Unfreeze top N layers
base_model.trainable = True
for layer in base_model.layers[:-UNFREEZE_LAYERS]:
    layer.trainable = False

# CRITICAL STABILITY FIX: Keep BatchNormalization layers frozen during fine-tuning
# to preserve pre-trained internal statistics (prevents model crashes and gradients exploding)
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

unfrozen = sum(1 for l in base_model.layers if l.trainable)
print(f'Unfrozen layers (Phase 2): {unfrozen}/{len(base_model.layers)}')

# CosineDecay LR
total_steps_p2 = EPOCHS_FINE * train_batch_count
cosine_decay_p2 = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=LR_FINE,
    decay_steps=total_steps_p2,
    alpha=1e-7
)

model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=cosine_decay_p2,
        weight_decay=WEIGHT_DECAY
    ),
    loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.05),
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

# Resume state
current_state = load_state()
p2_ckpt, p2_ckpt_epoch = find_latest_checkpoint('p2')

if current_state['phase'] == 'p2':
    p2_initial_epoch = current_state['last_epoch']
elif current_state['phase'] == 'done':
    p2_initial_epoch = EPOCHS_FINE
else:
    p2_initial_epoch = 0

history_p2 = None

if p2_initial_epoch >= EPOCHS_FINE:
    print(f'✅ Phase 2 already complete ({p2_initial_epoch}/{EPOCHS_FINE} epochs).')
    if p2_ckpt:
        model.load_weights(p2_ckpt)
        print(f'   Loaded weights: {os.path.basename(p2_ckpt)}')
else:
    if p2_ckpt and p2_initial_epoch > 0:
        model.load_weights(p2_ckpt)
        print(f'Resuming Phase 2 from epoch {p2_initial_epoch}')
    elif p2_ckpt and p2_ckpt_epoch > 0:
        model.load_weights(p2_ckpt)
        p2_initial_epoch = p2_ckpt_epoch
        print(f'Resuming Phase 2 from checkpoint epoch {p2_initial_epoch}')
    else:
        print('Starting Phase 2 from scratch.')

    cb_p2 = [
        callbacks.ModelCheckpoint(
            filepath=os.path.join(CHECKPOINT_DIR, 'artifact_p2_epoch_{epoch:02d}.keras'),
            save_best_only=True,
            monitor='val_accuracy',
            mode='max',
            verbose=1
        ),
        callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=6,
            restore_best_weights=True,
            verbose=1
        ),
        DriveStateLogger(phase='p2'),
        RAMMonitor(),
    ]

    print(f'\n=== PHASE 2: Fine-tune Top {UNFREEZE_LAYERS} Layers ===')
    print(f'    CosineDecay LR={LR_FINE} | Epochs: {p2_initial_epoch+1} -> {EPOCHS_FINE}')
    history_p2 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_FINE,
        initial_epoch=p2_initial_epoch,
        callbacks=cb_p2
    )

    save_state('done', EPOCHS_FINE)
    gc.collect()
    print('✅ Phase 2 complete.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 10: Evaluation on Test Set
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import json
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Recover class_names if not in memory (e.g., partial re-run)
if 'class_names' not in dir():
    with open(CLASS_NAMES_FILE, 'r') as f:
        class_names = json.load(f)
    print(f'Loaded class_names from Drive: {class_names}')

print('Evaluating on test set...')
results = model.evaluate(test_ds, verbose=1)
metric_names = model.metrics_names

print('\n' + '=' * 40)
print('TEST RESULTS')
print('=' * 40)
for name, val in zip(metric_names, results):
    pct = f' ({val*100:.2f}%)' if val <= 1.0 else ''
    print(f'   {name:12s}: {val:.4f}{pct}')

print('\nGenerating predictions...')
y_pred_probs = model.predict(test_ds, verbose=1)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()
y_true = np.concatenate([y for _, y in test_ds], axis=0).astype(int).flatten()

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - ArtiFact EfficientNetV2B0', fontsize=13, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
cm_path = os.path.join(DRIVE_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=150)
plt.show()
print(f'✅ Confusion matrix saved to: {cm_path}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 11: Save Final Model to Google Drive
# ─────────────────────────────────────────────────────────────────────────────
import json

model.save(MODEL_SAVE_PATH)
print(f'✅ Model saved to Drive: {MODEL_SAVE_PATH}')

import pickle
pkl_path = MODEL_SAVE_PATH.replace('.keras', '.pkl')
try:
    with open(pkl_path, 'wb') as f:
        pickle.dump(model, f)
    print(f'✅ Pickle saved: {pkl_path}')
except Exception as e:
    print(f'Pickle failed (not critical): {e}')

history_data = {}
if history_p1:
    history_data['p1'] = {
        k: [float(v) for v in vs]
        for k, vs in history_p1.history.items()
    }
if history_p2:
    history_data['p2'] = {
        k: [float(v) for v in vs]
        for k, vs in history_p2.history.items()
    }

with open(HISTORY_FILE, 'w') as f:
    json.dump(history_data, f, indent=2)
print(f'✅ Training history saved: {HISTORY_FILE}')

print(f'\nALL DONE!')
print(f'   Final model : {MODEL_SAVE_PATH}')
print(f'   Model size  : {os.path.getsize(MODEL_SAVE_PATH) / 1024 / 1024:.1f} MB')
print('\nNext steps:')
print('  1. Download artifact_efficientnetv2b0.keras from Google Drive')
print('  2. Place it in your project: models/ directory')
print('  3. Update model_loader.py to load this new model')
print('  4. Restart your backend Python API')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 12: Training History Plot
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

all_acc, all_val_acc = [], []
all_loss, all_val_loss = [], []
phase_boundaries = []

for h_obj in [history_p1, history_p2]:
    if h_obj:
        phase_boundaries.append(len(all_acc))
        all_acc      += h_obj.history.get('accuracy', [])
        all_val_acc  += h_obj.history.get('val_accuracy', [])
        all_loss     += h_obj.history.get('loss', [])
        all_val_loss += h_obj.history.get('val_loss', [])

if not all_acc:
    print('No history available in this session (all phases resumed/skipped).')
    print('Check training_history.json on Drive for full metrics.')
else:
    epochs = range(1, len(all_acc) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, all_acc,     'b-o', ms=3, label='Train Accuracy')
    ax1.plot(epochs, all_val_acc, 'r-o', ms=3, label='Val Accuracy')
    for i, b in enumerate(phase_boundaries[1:]):
        ax1.axvline(x=b, color='gray', linestyle='--', alpha=0.6,
                    label='Fine-tune start' if i == 0 else '_nolegend_')
    ax1.set_title('Accuracy Over Epochs', fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, all_loss,     'b-o', ms=3, label='Train Loss')
    ax2.plot(epochs, all_val_loss, 'r-o', ms=3, label='Val Loss')
    for i, b in enumerate(phase_boundaries[1:]):
        ax2.axvline(x=b, color='gray', linestyle='--', alpha=0.6,
                    label='Fine-tune start' if i == 0 else '_nolegend_')
    ax2.set_title('Loss Over Epochs', fontweight='bold')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.suptitle('ArtiFact EfficientNetV2B0 Training History', fontweight='bold', fontsize=14)
    plt.tight_layout()
    plt.savefig(PLOT_PATH, dpi=150)
    plt.show()
    print(f'✅ Plot saved to: {PLOT_PATH}')